In [66]:
import numpy as np
import torch
import torch.nn as nn

graph = np.load("model/graph.npy", allow_pickle=True).item()
operations = np.load("model/operations.npy", allow_pickle=True).tolist()
config = np.load("model/config.npy", allow_pickle=True).tolist()
weights = np.load("model/weights.npy", allow_pickle=True).tolist()
tensors = []

In [67]:
def tensor_vis(tensor, max_elements=1, indent=0):
    if not isinstance(tensor, torch.Tensor):
        raise ValueError("Input must be a torch.Tensor")
    shape = tensor.shape
    print(" " * indent + f"Tensor shape: {shape}")
    if tensor.ndim == 0:
        print(" " * indent + f"Value: {tensor.item()}")
        return
    elif tensor.ndim == 1:
        elems = tensor[:max_elements].tolist()
        if len(tensor) > max_elements:
            elems.append("...")
        print(" " * indent + str(elems))
    else:
        for i in range(min(tensor.shape[0], max_elements)):
            print(" " * indent + f"[{i}]:")
            tensor_vis(tensor[i], max_elements=max_elements, indent=indent + 2)
        if tensor.shape[0] > max_elements:
            print(" " * indent + "...")
            
print("Graph type:", type(graph))
print("Graph example (first 5 nodes):",graph)

# print("Operations type:", type(operations))
# print("Operations (first 10):", operations[:10])

# print("Config type:", type(config))
# print("Config (first 5):", config[:5])

# print("Weights type:", type(weights))
# print("Weights (first 3):", weights[:3])

Graph type: <class 'dict'>
Graph example (first 5 nodes): {0: [-1], 1: [0], 2: [1], 3: [2], 4: [3], 5: [4], 6: [5], 7: [6], 8: [7], 9: [8], 10: [9], 11: [10], 12: [11], 13: [12], 14: [13], 15: [14], 16: [15], 17: [16], 18: [17], 19: [18], 20: [19], 21: [20], 22: [21], 23: [22], 24: [23], 25: [24], 26: [25], 27: [26], 28: [27], 29: [28], 30: [29], 31: [30], 32: [31], 33: [32], 34: [33], 35: [34], 36: [35], 37: [36], 38: [37], 39: [38], 40: [39], 41: [40], 42: [41], 43: [42], 44: [43], 45: [44], 46: [45], 47: [46], 48: [47], 49: [48], 50: [49], 51: [50], 52: [51], 53: [52], 54: [53], 55: [54], 56: [55], 57: [56], 58: [57], 59: [58], 60: [59], 61: [60], 62: [61], 63: [62], 64: [63], 65: [64], 66: [65], 67: [66], 68: [67], 69: [68], 70: [69], 71: [70], 72: [71], 73: [72], 74: [73], 75: [74], 76: [75], 77: [76], 78: [77], 79: [78], 80: [79], 81: [80], 82: [81], 83: [82], 84: [83], 85: [84], 86: [85], 87: [86], 88: [87], 89: [88], 90: [89], 91: [90], 92: [91], 93: [92], 94: [93], 95: [94], 9

In [68]:
def Conv2d(config, weights):
    print("Config = [in_ch, out_ch, k_h, k_w, stride_h, stride_w, pad_h, pad_w, bias_flag]")
    print("Weight = [weight(out_ch x in_ch x kH x kW), bias(out_ch) if exists]")
    in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias_flag = config
    layer = nn.Conv2d(
        in_channels=in_ch,
        out_channels=out_ch,
        kernel_size=(k_h, k_w),
        stride=(s_h, s_w),
        padding=(p_h, p_w),
        bias=bias_flag
    )
    if weights:
        layer.weight.data = weights[0]
        if bias_flag and len(weights) > 1:
            layer.bias.data = weights[1]
    return layer

def BatchNorm2d(config, weights):
    print("Config = [num_features, eps, momentum, affine_flag, track_running_stats_flag]")
    print("Weight = [weight(num_features), bias(num_features), running_mean(num_features), running_var(num_features)]")
    num_feat, eps, momentum, affine_flag, track_stats = config
    layer = nn.BatchNorm2d(
        num_features=num_feat,
        eps=eps,
        momentum=momentum,
        affine=affine_flag,
        track_running_stats=track_stats
    )
    if weights:
        layer.weight.data = weights[0]
        layer.bias.data = weights[1]
        layer.running_mean.data = weights[2]
        layer.running_var.data = weights[3]
    return layer

def SiLU(config, weights):
    print("Config = [inplace_flag]")
    inplace_flag = config[0]
    return nn.SiLU(inplace=inplace_flag)

def MaxPool2d(config, weights):
    print("Config = [kernel_h, kernel_w, stride_h, stride_w, pad_h, pad_w, dilation_h, dilation_w, ceil_mode_flag]")
    k_h, k_w, s_h, s_w, p_h, p_w, d_h, d_w, ceil_mode = config
    return nn.MaxPool2d(
        kernel_size=(k_h, k_w),
        stride=(s_h, s_w),
        padding=(p_h, p_w),
        dilation=(d_h, d_w),
        ceil_mode=ceil_mode
    )

def Upsample(config, weights):
    print("Config = [scale_h, scale_w, mode]")
    scale_h, scale_w, mode = config
    return nn.Upsample(
        scale_factor=(scale_h, scale_w),
        mode=mode
    )

# Concat:
# Config = []
# Weight = []

def layer(operation, config, weights=None):
    if operation == "Conv2d":
        return Conv2d(config, weights)
    elif operation == "BatchNorm2d":
        return BatchNorm2d(config, weights)
    elif operation == "SiLU":
        return SiLU(config, weights)
    elif operation == "MaxPool2d":
        return MaxPool2d(config, weights)
    elif operation == "Upsample":
        return Upsample(config, weights)
    elif operation == "Concat":
        raise ValueError(f"Unsupported layer type: {operation}")
    else:
        raise ValueError(f"Unsupported layer type: {operation}")


In [69]:
from PIL import Image
import torch
from torchvision import transforms

def preprocess(image_path, img_size=(640, 640), device='cpu'):
    if isinstance(image_path, str):
        img = Image.open(image_path).convert("RGB")
    elif isinstance(image_path, Image.Image):
        img = image_path.convert("RGB")
    else:
        raise TypeError("image_path must be a file path or PIL.Image")

    transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
    ])

    tensor = transform(img).unsqueeze(0).to(device)
    return tensor

In [70]:
from PIL import Image
frame = Image.open("/mnt/fileserver/prj/yolo-inference-from-scratch/imgs/000000000785.jpg")
frame = preprocess(frame)
tensors.append(frame)

In [71]:
def step(l):
    print(f"\n>>> Layer {l} Info <<<")    
    print(f"\n-----in-----")
    current_layer = layer(operations[l], config[l], weights[l])
    input_tensor = tensors[graph[l][0]+1]
    print("layert:", operations[l])
    print(f"Input tensor shape: {input_tensor.shape}")
    print(f"Layer config: {config[l]}")
    print(f"Graph mapping: {graph[l]}")
    if weights[l]:
        for i, ts in enumerate(weights[l]):
            print(f"\nWeight tensor {i}:")
            tensor_vis(ts)
    
    output_tensor = current_layer(input_tensor)
    tensors.append(output_tensor)


    print(f"\n-----out-----")
    print("\nOutput tensor:")
    tensor_vis(output_tensor)

In [72]:
for l in range(400):
    try:
        step(l)
    except RuntimeError as e:
        print(f"\n>>> Layer {l} skipped due to error:")
        print(e)
        if tensors:
            for i, ts in enumerate(tensors):
                print(f"\nTensors {i}:")
                tensor_vis(ts)



>>> Layer 0 Info <<<

-----in-----
Config = [in_ch, out_ch, k_h, k_w, stride_h, stride_w, pad_h, pad_w, bias_flag]
Weight = [weight(out_ch x in_ch x kH x kW), bias(out_ch) if exists]
layert: Conv2d
Input tensor shape: torch.Size([1, 3, 640, 640])
Layer config: [3, 16, 6, 6, 2, 2, 2, 2, False]
Graph mapping: [-1]

Weight tensor 0:
Tensor shape: torch.Size([16, 3, 6, 6])
[0]:
  Tensor shape: torch.Size([3, 6, 6])
  [0]:
    Tensor shape: torch.Size([6, 6])
    [0]:
      Tensor shape: torch.Size([6])
      [-0.006977081298828125, '...']
    ...
  ...
...

-----out-----

Output tensor:
Tensor shape: torch.Size([1, 16, 320, 320])
[0]:
  Tensor shape: torch.Size([16, 320, 320])
  [0]:
    Tensor shape: torch.Size([320, 320])
    [0]:
      Tensor shape: torch.Size([320])
      [-2.31850266456604, '...']
    ...
  ...

>>> Layer 1 Info <<<

-----in-----
Config = [num_features, eps, momentum, affine_flag, track_running_stats_flag]
Weight = [weight(num_features), bias(num_features), running_m

IndexError: list index out of range

In [ ]:
print(graph)